### 08. Building a Collaborative Filtering Recommendation Model

**Objective:** Build a **User-Based Collaborative Filtering** recommender that suggests TED Talks based on similar users' rating patterns.

**Input:**
- `../data/interaction/user_interactions.csv` (from Task 7)
- `../data/processed/ted_talks_cleaned.csv` (for talk titles)

**⚠ Simulated Data:** The interaction data was simulated in Task 7 for learning purposes. No real user data is used.

**Approach:**
1. Load interactions and build the user-item matrix.
2. Fill missing values with **0** (simplified — 0 means "not rated", not "rated 0").
3. Compute **cosine similarity** between all users.
4. For a target user, find **top-K similar users** (K=10).
5. Predict scores for unrated talks using similarity-weighted average of neighbours' ratings.
6. Return the **top-N** recommendations.

In [43]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

##### Load the Interaction Dataset

In [44]:
interactions = pd.read_csv("../data/interaction/user_interactions.csv")
talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")[["talk_id", "title"]]

print(f"Interactions: {interactions.shape[0]} rows")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique talks: {interactions['talk_id'].nunique()}")
print(f"Rating range: {interactions['rating'].min()} – {interactions['rating'].max()}")
interactions.head()

Interactions: 1153 rows
Unique users: 50
Unique talks: 200
Rating range: 1 – 5


,user_id,talk_id,rating,liked,saved,watch_pct,timestamp
0,U001,1307,4,True,False,0.57,2023-07-02
1,U001,1075,3,False,False,0.62,2024-09-09
2,U001,2030,2,False,False,0.80,2023-12-01
3,U001,719,3,False,False,0.60,2023-10-15
4,U001,21894,2,False,False,0.43,2024-02-03


##### Build the User-Item Interaction Matrix
- **Rows:** users
- **Columns:** talks
- **Values:** rating (1–5)
- **NaN** = user has not rated the talk

In [45]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
)

print(f"Matrix shape: {user_item_matrix.shape}  (users × talks)")
print(f"Filled entries:  {user_item_matrix.notna().sum().sum()}")
print(f"Missing entries: {user_item_matrix.isna().sum().sum()}")
print(f"Sparsity:        {user_item_matrix.isna().mean().mean():.2%}")

user_item_matrix.iloc[:8, :8]

Matrix shape: (50, 200)  (users × talks)
Filled entries:  1153
Missing entries: 8847
Sparsity:        88.47%


talk_id,27,44,46,71,79,91,131,190
user_id,,,,,,,,
U001,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
U002,NaN,NaN,2.0,NaN,NaN,NaN,NaN,2.0
U003,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
U004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U005,NaN,NaN,NaN,4.0,3.0,NaN,NaN,NaN
U006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U007,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN
U008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0


##### Calculate User Similarity (Cosine)

For cosine similarity, we **fill missing ratings with 0**. This means "user X didn't rate this talk" — it does **not** mean they rated it 0.

Two users with similar non-zero rating patterns will have high cosine similarity.

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill missing with 0 for cosine similarity
filled_matrix = user_item_matrix.fillna(0).values

# Compute user-user similarity
user_sim = cosine_similarity(filled_matrix, dense_output=True)
np.fill_diagonal(user_sim, 0.0)  # exclude self

user_sim_df = pd.DataFrame(
    user_sim,
    index=user_item_matrix.index,
    columns=user_item_matrix.index,
)

print(f"User similarity matrix shape: {user_sim_df.shape}")
print(f"Similarity range: {user_sim_df.values.min():.4f} – {user_sim_df.values.max():.4f}")
print(f"Average pairwise similarity: {user_sim_df.values[np.triu_indices_from(user_sim_df.values, k=1)].mean():.4f}")

# Preview the first 6x6 block
user_sim_df.iloc[:6, :6].round(3)

User similarity matrix shape: (50, 50)
Similarity range: 0.0000 – 0.3343
Average pairwise similarity: 0.1079


user_id,U001,U002,U003,U004,U005,U006
user_id,,,,,,
U001,0.000,0.084,0.198,0.069,0.103,0.152
U002,0.084,0.000,0.110,0.093,0.136,0.265
U003,0.198,0.110,0.000,0.086,0.333,0.000
U004,0.069,0.093,0.086,0.000,0.183,0.250
U005,0.103,0.136,0.333,0.183,0.000,0.152
U006,0.152,0.265,0.000,0.250,0.152,0.000


> **Note on low similarity scores:** With 88.5% sparsity, two users rarely rate the same talks, so most cosine similarities are small (avg ~0.11, max ~0.33). Values closer to 1.0 would only occur if users rated many of the *same* talks with similar scores. This is why CF recommendation quality depends heavily on data density.

##### Identify Similar Users for a Sample User
Let's look at User `U001`'s top-10 most similar users.

In [47]:
SAMPLE_USER = "U001"

sims = user_sim_df.loc[SAMPLE_USER].sort_values(ascending=False)
top_similar = sims.head(10)

print(f"Top 10 users most similar to {SAMPLE_USER}:")
print(top_similar.round(4))

Top 10 users most similar to U001:
user_id
U040    0.2738
U008    0.2652
U048    0.2518
U041    0.2462
U039    0.2328
U009    0.2308
U007    0.2216
U030    0.2066
U003    0.1979
U026    0.1968
Name: U001, dtype: float64


##### Recommendation Function

**`recommend_for_user(user_id, top_n=5, k_neighbors=10)`**

Steps:
1. Validate the user exists.
2. Take the top-K most similar users (neighbours).
3. For each talk the target user **has not rated**, compute a predicted score:
   $$\hat{r}_{u,i} = \frac{\sum_{v \in N} \text{sim}(u,v) \cdot r_{v,i}}{\sum_{v \in N} |\text{sim}(u,v)|}$$
   where N = neighbours who actually rated talk *i*.
4. Sort by predicted score, take top-N.
5. Attach talk titles.

In [48]:
def recommend_for_user(user_id, top_n=5, k_neighbors=10):
    """
    User-based collaborative filtering recommender.

    Parameters:
    - user_id (str): The user to recommend for.
    - top_n (int): Number of recommendations (default 5).
    - k_neighbors (int): Number of similar users to consider (default 10).

    Returns:
    - pd.DataFrame: columns [Rank, talk_id, title, Predicted Score]
    - None: if user is invalid or has no interactions.
    """
    # Validate user
    if user_id not in user_item_matrix.index:
        print(f"Invalid user_id: '{user_id}'. Not found in the interaction dataset.")
        return None

    # Validate top_n
    if not isinstance(top_n, int) or top_n < 1 or top_n > 20:
        print("Error: top_n must be an integer between 1 and 20.")
        return None

    # Talks the target user has already rated
    user_ratings = user_item_matrix.loc[user_id]
    already_rated = user_ratings.dropna().index
    if len(already_rated) == 0:
        print(f"User '{user_id}' has no interaction history — cold-start. Cannot compute CF recommendations.")
        return None

    # Similarity scores to all other users
    sims = user_sim_df.loc[user_id].drop(user_id)

    # Top-K neighbours
    neighbours = sims.sort_values(ascending=False).head(k_neighbors)
    neighbour_ids = neighbours.index
    neighbour_weights = neighbours.values

    # Candidate talks: not rated by target user, rated by at least one neighbour
    neighbour_ratings = user_item_matrix.loc[neighbour_ids]
    candidate_mask = user_ratings.isna()
    neighbours_with_data = neighbour_ratings.notna()

    # Weighted average predictions
    preds = {}
    for talk_id in user_item_matrix.columns:
        if not candidate_mask[talk_id]:
            continue  # user already rated it
        rated_by = neighbours_with_data[talk_id]
        if not rated_by.any():
            continue

        weights = neighbour_weights[rated_by.values]
        ratings = neighbour_ratings.loc[rated_by, talk_id].values

        denom = weights.sum()
        if denom <= 0:
            continue
        preds[talk_id] = float(np.dot(weights, ratings) / denom)

    if not preds:
        print(f"No recommendations could be generated for '{user_id}'.")
        return None

    # # Sort and take top-N
    # pred_series = pd.Series(preds).sort_values(ascending=False).head(top_n)
    # rec_df = pred_series.reset_index()
    # rec_df.columns = ["talk_id", "Predicted Score"]
    # rec_df.insert(0, "Rank", range(1, len(rec_df) + 1))
    # rec_df["Predicted Score"] = rec_df["Predicted Score"].round(4)

    # Sort and take top-N (tiebreak by number of neighbours who rated)
    pred_data = []
    for talk_id, score in preds.items():
        rated_by = neighbours_with_data[talk_id]
        n_raters = int(rated_by.sum())
        pred_data.append((talk_id, score, n_raters))

    pred_df = pd.DataFrame(pred_data, columns=["talk_id", "Predicted Score", "n_raters"])
    pred_df = pred_df.sort_values(["Predicted Score", "n_raters"], ascending=[False, False]).head(top_n)
    pred_df.insert(0, "Rank", range(1, len(pred_df) + 1))
    pred_df["Predicted Score"] = pred_df["Predicted Score"].round(4)

    # Attach titles
    rec_df = pred_df[["Rank", "talk_id", "Predicted Score"]]
    rec_df = rec_df.merge(talks, on="talk_id", how="left")
    rec_df = rec_df[["Rank", "talk_id", "title", "Predicted Score"]]
    return rec_df

##### Test the Recommendation Function

Test with three different users and one invalid user.

In [49]:
# Test 1 — U001
recommend_for_user("U001", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,386,the story of ezra,4.0
1,2,27,organic design inspired by nature,4.0
2,3,2368,forget wi fi meet the new li fi internet,4.0
3,4,2883,lessons from the longest study on human development,4.0
4,5,32269,the uncomplicated truth about women s sexuality,4.0


>**Note**: U001's top-5 all score 4.0 because all its neighbours gave those talks identical ratings — the tiebreaker (n_raters) breaks the tie deterministically.

In [50]:
# Test 2 — U006 (one of the most active users in Task 7)
recommend_for_user("U006", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,2265,the surprisingly logical minds of babies,5.0
1,2,27,organic design inspired by nature,4.0
2,3,246,inventing instruments that unlock new music,4.0
3,4,1656,a new kind of job market,4.0
4,5,1993,comics that ask what if,4.0


In [51]:
# Test 3 — U042
recommend_for_user("U042", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,1140,the hidden beauty of pollination,4.5193
1,2,246,inventing instruments that unlock new music,4.0000
2,3,359,the blur building and other tech empowered architecture,4.0000
3,4,675,compassion at the dinner table,4.0000
4,5,1800,the voices in my head,4.0000


In [52]:
# Test 4 — Invalid user
recommend_for_user("U999", top_n=5)

Invalid user_id: 'U999'. Not found in the interaction dataset.


##### Verification Checks

We verify the function meets the four required checks:
1. Generates recommendations for valid users.
2. Excludes talks already rated by the user.
3. Returns sorted results.
4. Different users get different recommendations.

In [53]:
def verify_recommendations(user_id, top_n=5):
    print(f"\nVerifying recommendations for {user_id}...")
    recs = recommend_for_user(user_id, top_n=top_n)
    if recs is None:
        return

    # 1. Excludes already-rated talks
    rated = set(user_item_matrix.loc[user_id].dropna().index)
    overlap = set(recs["talk_id"]) & rated
    print(f"  ✔ Talks already rated by user: {len(overlap)} (expected 0)")

    # 2. Sorted by predicted score
    scores = recs["Predicted Score"].values
    is_sorted = all(scores[i] >= scores[i + 1] for i in range(len(scores) - 1))
    print(f"  ✔ Sorted by predicted score: {is_sorted}")

    # 3. Returns exactly top_n rows
    print(f"  ✔ Returned rows: {len(recs)} (requested {top_n})")

for uid in ["U001", "U006", "U042"]:
    verify_recommendations(uid, top_n=5)


Verifying recommendations for U001...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)

Verifying recommendations for U006...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)

Verifying recommendations for U042...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)


In [54]:
# 4. Different users → different recommendations
rec_a = recommend_for_user("U001", top_n=5)
rec_b = recommend_for_user("U006", top_n=5)

overlap_ids = set(rec_a["talk_id"]) & set(rec_b["talk_id"])
print(f"Overlap between U001 and U006 recommendations: {len(overlap_ids)} talks")
print(f"U001 got: {rec_a['talk_id'].tolist()}")
print(f"U006 got: {rec_b['talk_id'].tolist()}")

Overlap between U001 and U006 recommendations: 1 talks
U001 got: [386, 27, 2368, 2883, 32269]
U006 got: [2265, 27, 246, 1656, 1993]


##### Answers to Required Questions

**1. What is Collaborative Filtering?**
A recommendation technique that predicts a user's preferences based on the behaviour of *other similar users* — without analysing the content of the items. "Users like you liked X, so you might too."

**2. How is it different from Content-Based Filtering?**
| Aspect | Content-Based | Collaborative |
|---|---|---|
| Uses | Item features (title, tags, transcript) | User-item interactions (ratings) |
| Asks | "What is this talk about?" | "Who else liked this talk?" |
| Personalisation | Per-user preferences on item features | Per-user preferences on other users' behaviour |
| New item | Works well (features available) | Fails (no ratings yet — cold item) |
| New user | Works (needs a seed item) | Fails (no history — cold user) |

**3. What is a User-Item Interaction Matrix?**
A 2-D table where **rows = users, columns = talks, values = ratings**. A missing value means "user did not rate that talk."

**4. Why do we calculate similarity between users?**
To find users whose rating patterns match the target user. Their ratings are then used to predict scores for talks the target user hasn't seen.

**5. What does a cosine similarity score represent?**
The **angle** between two users' rating vectors. Range 0–1:
- 1.0 = identical rating patterns
- 0.0 = no shared rating behaviour
Because we fill missing values with 0, only talks rated by *both* users meaningfully contribute.

**6. Why should we exclude talks the user already rated?**
Recommending something they've already seen is useless. CF should only surface *new* items.

**7. What is a predicted rating?**
An estimate of how much the target user would rate a given talk, computed as the similarity-weighted average of ratings from their top-K similar users.

**8. What happens when a new user has no interaction history?**
The function returns an error / `None`. There's no data to compute similarity or predictions — the standard **cold-start** failure mode.

**9. What is the Cold-Start Problem?**
The inability to make good recommendations when:
- **New user:** no rating history → no similarity signal.
- **New item:** no ratings → never recommended.
Typical mitigations: ask for onboarding preferences, fall back to popularity, or use content-based features (hybrid).

**10. What are the limitations of User-Based CF?**
- **Cold start** for new users and new items.
- **Sparsity:** most user-user pairs share very few rated items → unstable similarity.
- **Scalability:** O(users²) similarity computation — expensive at millions of users.
- **Popularity bias:** popular talks dominate; niche tastes are under-served.
- **Shilling / manipulation:** fake ratings can manipulate neighbours.
- **User tastes change over time** — similarity based on old ratings may go stale.
- **Zero-fill loses information:** treats unrated as "rated 0", biasing similarity downward for sparse pairs.

##### Observations

1. **Similar users are identified** by cosine similarity on their rating vectors (missing values → 0). Two users who both rate the same talks high will get a high similarity score.
2. **Recommendations are generated** as a similarity-weighted average of ratings from the target's top-K (default 10) neighbours, restricted to talks the target hasn't rated.
3. **Different users receive different recommendations.** U001, U006, and U042 each have 
   different neighbour sets and different rated talks, so their top-N lists diverge. 
   U001 and U006 share only 1 recommendation out of 5 (talk_id=27) — likely a popular 
   talk that appears across many neighbourhoods.
4. **User preferences directly influence results** — a user whose ratings overlap strongly with a neighbour inherits that neighbour's tastes. Conversely, users with few shared ratings may get noisier predictions.
5. **Limitations observed:**
   - **Cold-start:** users with zero ratings can't get CF recommendations (handled with an error + fallback message).
   - **Sparsity:** with 88% missing data, most user-user pairs share very few co-rated talks, so similarity scores are modest and predictions rely on a handful of neighbours.
   - **Zero-fill approximation:** treating NaN as 0 penalises users who only rated a small subset — it artificially lowers similarity even when their few shared ratings agree.
   - **Popularity bias:** talks rated by many users are more likely to appear in predictions simply because they have data.
   - **No item features used:** two very different talks with similar rating patterns look identical to the model.
6. **Next step (Task 9):** Combine this CF model with the content-based recommender from Task 6 into a **hybrid** system — CF handles personalisation, content handles cold-start and diverse item descriptions.

---

## 📄 Task Report

The written report for this notebook lives at [`reports/task_08_observations.md`](../reports/task_08_observations.md).

- Markdown: [`reports/task_08_observations.md`](../reports/task_08_observations.md)
- PDF: [`reports/pdf/task_08_observations.pdf`](../reports/pdf/task_08_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
